# Rootline: score the two small models on a Kaggle GPU

Runs Qwen3 8B and Gemma 3 4B (Ollama, default 4-bit) on every snapshot and
evidence condition, then uploads the runs to the private Hugging Face
dataset `Flotrpy/rootline-work`.

Before running:
1. **Settings → Accelerator → GPU T4 x2**, and **Settings → Internet → On** (needs a phone-verified Kaggle account).
2. **Add-ons → Secrets → Add secret** named `HF_TOKEN` with your Hugging Face write token, and switch it on for this notebook.
3. **Run all.** It takes about 2 hours; the queue resumes if the session restarts.


In [ ]:
# 1. GPU and Ollama
!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)
!ollama pull qwen3:8b
!ollama pull gemma3:4b
!ollama list

In [ ]:
# 2. Code and data (snapshots and labels come from the private Hugging Face dataset)
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
%cd /kaggle/working
!test -d RootLine || git clone -q --branch claude/optimistic-johnson-tp1lxw https://github.com/Flotrpy/RootLine.git
%cd /kaggle/working/RootLine
!git pull -q
!pip -q install uv
!uv run --quiet --project harness python -m analysis fetch-data

In [ ]:
# 3. Score (one queue file; rerunning this cell resumes where it stopped)
Q = "--db data/queue-ollama.sqlite"
!uv run --quiet --project harness python -m rootline_harness.run {Q} enqueue --models qwen3-8b-q4@ollama gemma3-4b-q4@ollama
!uv run --quiet --project harness python -m rootline_harness.run {Q} work
!uv run --quiet --project harness python -m rootline_harness.run {Q} status

In [ ]:
# 4. Export and upload to the private work dataset
!uv run --quiet --project harness python -m rootline_harness.run {Q} export --out data/runs/runs-ollama.parquet
import subprocess
from huggingface_hub import HfApi
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout
open("data/runs/ollama-hardware.txt", "w").write(gpu)
api = HfApi()  # uses HF_TOKEN from the Kaggle secret
assert api.repo_info("Flotrpy/rootline-work", repo_type="dataset").private, "work dataset must stay private"
for f in ["data/runs/runs-ollama.parquet", "data/queue-ollama.sqlite", "data/runs/ollama-hardware.txt"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=f.split("/")[-1], repo_id="Flotrpy/rootline-work",
                    repo_type="dataset", commit_message="Kaggle GPU: small-model runs")
print("uploaded")